In [1]:
import matplotlib.pyplot as plt 
import torch 
import torch.nn as nn
import torch.nn.functional as F
from torchvision import datasets, transforms
from torch.utils.data import DataLoader , TensorDataset


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)




cuda


In [2]:
transform = transforms.Compose([ transforms.Resize((64,64)), 
                                transforms.ToTensor(), 
                                transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5])])

from torchvision.datasets import CIFAR10

dataset = CIFAR10(
    root="./data",
    train=True,
    download=True,
    transform=transform
)

from torch.utils.data import DataLoader

loader = DataLoader(
    dataset,
    batch_size=64,
    shuffle=True
)


100%|██████████| 170M/170M [00:03<00:00, 48.0MB/s] 


In [15]:
class Generator(nn.Module):
    def __init__(self, z_dim=100, channels_img=3, features_g=64):
        super().__init__()

        self.net = nn.Sequential(

            nn.ConvTranspose2d(
                z_dim,
                features_g * 8,
                kernel_size=4,
                stride=1,
                padding=0
            ),
            nn.BatchNorm2d(features_g * 8),
            nn.ReLU(True),

            nn.ConvTranspose2d(
                features_g * 8,
                features_g * 4,
                kernel_size=4,
                stride=2,
                padding=1
            ),
            nn.BatchNorm2d(features_g * 4),
            nn.ReLU(True),

            nn.ConvTranspose2d(
                features_g * 4,
                features_g * 2,
                kernel_size=4,
                stride=2,
                padding=1
            ),
            nn.BatchNorm2d(features_g * 2),
            nn.ReLU(True),

            nn.ConvTranspose2d(
                features_g * 2,
                features_g,
                kernel_size=4,
                stride=2,
                padding=1
            ),
            nn.BatchNorm2d(features_g),
            nn.ReLU(True),

            nn.ConvTranspose2d(
                features_g,
                channels_img,
                kernel_size=4,
                stride=2,
                padding=1
            ),
            nn.Tanh()
        )

    def forward(self, x):
        return self.net(x)
    
class Critic(nn.Module):
    def __init__(self, channels_img=3, features_d=64):
        super().__init__()

        self.net = nn.Sequential(

            nn.Conv2d(
                channels_img,
                features_d,
                kernel_size=4,
                stride=2,
                padding=1
            ),
            nn.LeakyReLU(0.2),

            nn.Conv2d(
                features_d,
                features_d * 2,
                kernel_size=4,
                stride=2,
                padding=1
            ),
            nn.BatchNorm2d(features_d * 2),
            nn.LeakyReLU(0.2),

            nn.Conv2d(
                features_d * 2,
                features_d * 4,
                kernel_size=4,
                stride=2,
                padding=1
            ),
            nn.BatchNorm2d(features_d * 4),
            nn.LeakyReLU(0.2),

            nn.Conv2d(
                features_d * 4,
                features_d * 8,
                kernel_size=4,
                stride=2,
                padding=1
            ),
            nn.BatchNorm2d(features_d * 8),
            nn.LeakyReLU(0.2),

            nn.Conv2d(
                features_d * 8,
                1,
                kernel_size=4,
                stride=1,
                padding=0
            ),
            nn.Tanh()

        
        )

    def forward(self, x):
        return self.net(x)
    
def initialize_weights(model):
     for m in model.modules():

        if isinstance(m, (nn.Conv2d,
                          nn.ConvTranspose2d,
                          nn.BatchNorm2d)):
            nn.init.normal_(m.weight.data, 0.0, 0.02)
            
            
def gradient_penalty(critic,real,fake,device):

    B,C,H,W = real.shape

    epsilon = torch.rand(
     real.size(0),
     1,
     1,
     1,
     device=device)

    interpolated = ( epsilon * real + (1-epsilon) * fake)

    interpolated.requires_grad_(True)

    mixed_scores = critic(interpolated)

    gradient = torch.autograd.grad(
        inputs=interpolated,
        outputs=mixed_scores,
        grad_outputs=torch.ones_like(mixed_scores),
        create_graph=True,
        retain_graph=True
    )[0]

    gradient = gradient.reshape(
        gradient.shape[0],
        -1
    )

    gradient_norm = gradient.norm(
        2,
        dim=1
    )

    gp = torch.mean(
        (gradient_norm - 1)**2
    )

    return gp

In [16]:
device = "cuda" if torch.cuda.is_available() else "cpu"

lr = 1e-4

batch_size = 128

z_dim = 128

num_epochs = 50

features_critic = 64

features_gen = 64

critic_iterations = 5

lambda_gp = 10

In [17]:
critic = Critic(
    channels_img=3,
    features_d=features_critic
).to(device)

gen = Generator(
    z_dim=z_dim,
    channels_img=3,
    features_g=features_gen
).to(device)

initialize_weights(critic)
initialize_weights(gen)

In [18]:
opt_critic = torch.optim.Adam(
    critic.parameters(),
    lr=lr,
    betas=(0.0, 0.9)
)

opt_gen = torch.optim.Adam(
    gen.parameters(),
    lr=lr,
    betas=(0.0, 0.9)
)

In [ ]:
for epoch in range(num_epochs):

    for batch_idx, (real, _) in enumerate(loader):

        real = real.to(device)

        cur_batch_size = real.shape[0]

        ##########################
        # Train Critic
        ##########################

        for _ in range(critic_iterations):

            noise = torch.randn(
                cur_batch_size,
                z_dim,
                1,
                1,
                device=device
            )

            fake = gen(noise)

            critic_real = critic(real).reshape(-1)

            critic_fake = critic(
                fake.detach()
            ).reshape(-1)

            gp = gradient_penalty(
                critic,
                real,
                fake.detach(),
                device
            )

            loss_critic = (
                -(torch.mean(critic_real)
                  -
                  torch.mean(critic_fake))
                +
                lambda_gp * gp
            )

            critic.zero_grad()

            loss_critic.backward()

            opt_critic.step()

        ##########################
        # Train Generator
        ##########################

        noise = torch.randn(
            cur_batch_size,
            z_dim,
            1,
            1,
            device=device
        )

        fake = gen(noise)

        gen_loss = -torch.mean(
            critic(fake)
        )

        gen.zero_grad()

        gen_loss.backward()

        opt_gen.step()

        if batch_idx % 100 == 0:

            print(
                f"Epoch [{epoch}/{num_epochs}] "
                f"Batch [{batch_idx}/{len(loader)}] "
                f"Critic Loss: {loss_critic:.4f} "
                f"Generator Loss: {gen_loss:.4f}"
            )

Epoch [0/50] Batch [0/782] Critic Loss: 5.5811 Generator Loss: 0.2774
Epoch [0/50] Batch [100/782] Critic Loss: -1.9478 Generator Loss: 0.9905
Epoch [0/50] Batch [200/782] Critic Loss: -1.9348 Generator Loss: 0.9922
Epoch [0/50] Batch [300/782] Critic Loss: -1.9368 Generator Loss: 0.9957
Epoch [0/50] Batch [400/782] Critic Loss: -1.9248 Generator Loss: 0.9968
Epoch [0/50] Batch [500/782] Critic Loss: -1.9343 Generator Loss: 0.9958
Epoch [0/50] Batch [600/782] Critic Loss: -1.8704 Generator Loss: 0.9990
Epoch [0/50] Batch [700/782] Critic Loss: -1.9399 Generator Loss: 0.9970
Epoch [1/50] Batch [0/782] Critic Loss: -1.9847 Generator Loss: 0.9996
Epoch [1/50] Batch [100/782] Critic Loss: -1.9880 Generator Loss: 0.9996
Epoch [1/50] Batch [200/782] Critic Loss: -1.9367 Generator Loss: 0.9993
Epoch [1/50] Batch [300/782] Critic Loss: -1.8308 Generator Loss: 0.9994
Epoch [1/50] Batch [400/782] Critic Loss: -1.9682 Generator Loss: 0.9998
Epoch [1/50] Batch [500/782] Critic Loss: -1.9838 Genera